# Проверка параллельной обработки Ollama

Этот тест **не строит граф**. Он отправляет четыре сохранённых фрагмента с текущим промптом извлечения в `qwen3.5-graph:latest`: один за другим и по два одновременно. Перед замером модель прогревается. Сравнивайте **время всей группы**, а не время одного запроса.

**Итог для установленной Ollama 0.34.2:** два клиентских запроса к Qwen 3.5 не дают параллельной генерации даже при `OLLAMA_NUM_PARALLEL=2`. В исходном коде этой версии архитектура `qwen35` явно ограничена одним слотом. Результаты двух повторов: 138,50 против 138,31 с и 141,32 против 135,79 с. Разница соответствует разбросу времени генерации. Дополнительная настройка переменной для этой модели не ускорит индексирование.

Ноутбук полезен для проверки **другой модели или сервера**. Он сохраняет время, число сгенерированных токенов и признак завершения по лимиту. Прямой вызов API близок к промпту GraphRAG, но не полностью повторяет его клиентскую библиотеку.

In [1]:
from pathlib import Path
from collections import Counter
import concurrent.futures, datetime, json, time, urllib.request

ROOT = next(p.resolve() for p in (Path.cwd(), Path.cwd().parent) if (p/'pyproject.toml').is_file())
RUN = ROOT/'local_runs/ganoshenko-hw3-3248caaa7ac4'
OUT = ROOT/'hw3_runs/ganoshenko'
MODEL = 'qwen3.5-graph:latest'
URL = 'http://127.0.0.1:11434'

chunks = [json.loads(line) for line in (OUT/'chunks.jsonl').read_text(encoding='utf-8').splitlines() if line]
selected = []
for target in (450, 600, 750, 850):
    options = (c for c in chunks if c['kind']=='text' and c['id'] not in {x['id'] for x in selected})
    selected.append(min(options, key=lambda c: abs(c['token_count']-target)))
template = (RUN/'prompts/extract_metallurgy.txt').read_text(encoding='utf-8')
entity_types = 'ХИМИЧЕСКИЙ_ЭЛЕМЕНТ, МАТЕРИАЛ, СОЕДИНЕНИЕ, МИКРОСТРУКТУРА, ТЕХНОЛОГИЧЕСКИЙ_ПРОЦЕСС, СВОЙСТВО'
prompts = [template.replace('{entity_types}',entity_types).replace('{input_text}',c['text']) for c in selected]
print([(c['id'], c['token_count']) for c in selected])

[('chunk-0079', 449), ('chunk-0040', 609), ('chunk-0106', 750), ('chunk-0087', 838)]


## Проверка кэша полного запуска

`finish_reason=length` означает, что ответ достиг лимита `max_tokens=3072`. Такой ответ может содержать лишь часть запрошенных записей графа. Отдельно считаем завершающий маркер промпта.

In [2]:
reasons = Counter()
complete = 0
cache_dir = RUN/'cache/extract_graph'
for path in cache_dir.iterdir():
    result = json.loads(path.read_text(encoding='utf-8'))['result']
    choice = result['response']['choices'][0]
    reasons[choice['finish_reason']] += 1
    complete += '<|COMPLETE|>' in (choice['message'].get('content') or '')
print('Ответов:', sum(reasons.values()), 'Причины завершения:', dict(reasons), 'С завершающим маркером:', complete)

Ответов: 111 Причины завершения: {'length': 84, 'stop': 27} С завершающим маркером: 27


## Замер на одинаковых четырёх фрагментах

Первый короткий запрос только загружает модель и не входит во время. `num_predict=3072` повторяет лимит текущего GraphRAG. Если все ответы обрезаны, результат применим к скорости, но не подтверждает качество извлечения.

In [3]:
def call_ollama(prompt: str) -> dict:
    payload = json.dumps({
        'model': MODEL,
        'messages': [{'role':'user','content':prompt}],
        'stream': False,
        'think': False,
        'keep_alive': '15m',
        'options': {'temperature':0.1, 'num_predict':3072, 'num_ctx':8192},
    }).encode('utf-8')
    request = urllib.request.Request(URL+'/api/chat', data=payload, headers={'Content-Type':'application/json'})
    started = time.perf_counter()
    with urllib.request.urlopen(request, timeout=600) as response:
        data = json.load(response)
    output = data['message'].get('content') or ''
    return {
        'wall_seconds': round(time.perf_counter()-started,2),
        'prompt_tokens': data.get('prompt_eval_count'),
        'generated_tokens': data.get('eval_count'),
        'finish_reason': data.get('done_reason'),
        'complete_marker': '<|COMPLETE|>' in output,
        'entities': output.count('("entity"<|>'),
        'relationships': output.count('("relationship"<|>'),
    }

call_ollama('Ответь одним словом: готово.')
with urllib.request.urlopen(URL+'/api/ps', timeout=10) as response:
    loaded = json.load(response).get('models',[])
print('Видеопамять модели, ГБ:', [(m.get('name'),round(m.get('size_vram',0)/1024**3,2)) for m in loaded])
results = {}
for concurrency in (1,2):
    start = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = [executor.submit(call_ollama,prompt) for prompt in prompts]
        rows = []
        for chunk, future in zip(selected, futures):
            rows.append({'chunk_id':chunk['id'], 'chunk_tokens':chunk['token_count'], **future.result()})
    results[str(concurrency)] = {'batch_wall_seconds':round(time.perf_counter()-start,2), 'rows':rows}
    print('Одновременных запросов:',concurrency,'время группы, с:',results[str(concurrency)]['batch_wall_seconds'])

speedup = results['1']['batch_wall_seconds']/results['2']['batch_wall_seconds']
print('Ускорение:',round(speedup,2),'×')
print('Обрезанные ответы:',{key:sum(row['finish_reason']=='length' for row in value['rows']) for key,value in results.items()})

stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
report_path = OUT/f'parallel_benchmark_{stamp}.json'
report_path.write_text(json.dumps({'model':MODEL,'chunks':[{'id':c['id'],'tokens':c['token_count']} for c in selected],
                                   'results':results,'speedup':round(speedup,3)},ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print('Отчёт:',report_path)

Видеопамять модели, ГБ: [('qwen3.5-graph:latest', 5.34)]
Одновременных запросов: 1 время группы, с: 138.5
Одновременных запросов: 2 время группы, с: 138.31
Ускорение: 1.0 ×
Обрезанные ответы: {'1': 3, '2': 4}
Отчёт: C:\Users\qa1ro\OneDrive\Рабочий стол\projects\graph_hw\hw3_runs\ganoshenko\parallel_benchmark_20260924-004426.json


**Как принять решение.** Для текущей Qwen 3.5 и Ollama 0.34.2 ожидайте около `1.0×`: сервер принудительно использует один слот. Повторять тест после изменения `OLLAMA_NUM_PARALLEL` для этой модели не требуется. Для другой модели или сервера сравнивайте общее время группы, расход видеопамяти и завершённость ответов. Ускорение на четырёх фрагментах — предварительный тест, не обещание такого же выигрыша за весь полный запуск.